In [3]:
import numpy as np
import pandas as pd
from loader import load_rentals, load_users

rentals = load_rentals()
users = load_users()


## [STEP 1] 파생변수 생성 및 정답지(Target) 만들기

### 요구사항

`clean_rentals.csv`를 불러온 뒤 아래 파생변수들을 만드세요.

1. **대여 시간(분)**: `duration_min = return_time - rent_time` 의 분 단위 계산
2. **`is_overdue_100m` (예측 대상)**: 대여 시간(`duration_min`)이 100분(1시간 40분)을 초과하면 `1`, 아니면 `0`
3. 대여 시점(`rent_time`) 기준 **시간대(hour)** 와 **요일** 추출
- 전체 행 수와 **`is_overdue_100m` 의 비율**을 확인하세요.
- **"전부 정상(미반납 아님)"이라고 답할 때의 정확도**를 계산하세요. 담당자가 말한 "정확도 84%"의 정체가 바로 이것입니다.

In [ ]:
# 파생변수 생성
rentals['duration_min'] = (rentals['return_time'] - rentals['rent_time']).dt.total_seconds() / 60
rentals['is_overdue_100m'] = (rentals['duration_min'] > 100).astype(int)
rentals['hour'] = rentals['rent_time'].dt.hour
rentals['dayofweek'] = rentals['rent_time'].dt.dayofweek

# 통계 및 비율 확인
total_rows = len(rentals)
overdue_ratio = rentals['is_overdue_100m'].mean()

print(f"전체 행 수: {total_rows:,}건")
print("is_overdue_100m 분포:")
print(f"초과(1) 비율 : {overdue_ratio * 100:.2f}%")
print(f"정상(0) 비율 : {(1-overdue_ratio) * 100:.2f}%")


전체 행 수: 13,212건
is_overdue_100m 분포:
초과(1) 비율 : 17.52%
정상(0) 비율 : 82.48%


,rental_id,rent_time,return_time,duration_min,is_overdue_100m,hour,weekday
0,R000002,2022-06-14 20:28:00,2022-06-14 21:29:00,61.0,0,20,1
1,R000004,2022-03-17 06:02:00,2022-03-17 06:38:00,36.0,0,6,3
2,R000005,2022-12-23 18:31:00,2022-12-23 19:27:00,56.0,0,18,4
3,R000006,2022-04-23 11:51:00,2022-04-23 13:25:00,94.0,0,11,5
4,R000007,2022-12-08 22:35:00,2022-12-08 23:56:00,81.0,0,22,3


## [STEP 2] 결합 및 사용자 파생변수

### 요구사항

- 대여 기록에 `users.csv` 를 결합하세요. (`how` 와 `validate` 명시)
- **결합 전후 행 수를 비교**하여 중복이 발생하지 않았는지 확인하세요.
- `join_date` 와 `rent_time` 을 활용해, 대여 시점 기준 **유저의 가입 기간(일)** 파생변수를 만드세요.

In [22]:
print(f"결합 전 rentals 행 수: {len(rentals):,}")

merged = rentals.merge(users, on='user_id', how='left', validate='many_to_one')

# 결합 후 행 수 확인
print(f"결합 후 merged 행 수: {len(merged):,}")

# join_date를 datetime 형식으로 변환 후, 대여 시점 기준 가입 기간(일) 계산
merged['join_date'] = pd.to_datetime(merged['join_date'])
merged['days_since_join'] = (merged['rent_time'] - merged['join_date']).dt.days

# 마이너스 값을 0으로 처리
merged.loc[merged['days_since_join'] < 0, 'days_since_join'] = 0

print(merged['days_since_join'].head(10))

결합 전 rentals 행 수: 13,212
결합 후 merged 행 수: 13,212
0      0
1      0
2    559
3      0
4      0
5      0
6     18
7    426
8    178
9      0
Name: days_since_join, dtype: int64


## [STEP 3] 데이터 누수 탐지

**이 단계가 이 문제의 첫 번째 관문입니다.**

지금 데이터프레임 안에는 **모델에 넣으면 안 되는 컬럼**이 섞여 있습니다.
학습 시점(대여 시점)에는 절대 알 수 없는 정보, 즉 **결과가 나온 뒤(반납 시점)에야 확정되는 값**들입니다.

### 요구사항

1. 각 컬럼들을 살펴보고 `is_overdue_100m` 와 직결되어 있는 누수(Leakage) 컬럼을 모두 찾아내세요.
2. 의심스러운 컬럼들을 찾고, **왜 학습에 쓰면 안 되는지** 설명하세요.

**판단 기준**

“이 값은 **언제** 확정되는가?”를 기준으로 판단해보세요.

우리는 사용자가 자전거를 **빌려가는 순간(`rent_time`)**에 미반납 여부를 예측해야 합니다. 그 시점에 알 수 없는 값(ex. 주행 거리)이 포함되어 있다면 모두 누수로 판단하면 됩니다.

**리포트에 반드시 쓸 것**

- 제거한 컬럼 이름들 (최소 3개 이상)
- 그 값이 **언제** 확정되는지
- 그대로 학습하면 실제 운영에서 무슨 일이 벌어지는지

In [23]:
# 누수(Leakage) 컬럼 제거
# 예측 시점(rent_time)에 알 수 없는 반납 관련 정보나 타겟 계산에 사용된 컬럼들을 피처셋에서 제외합니다.
leakage_cols = ['return_time', 'duration_min', 'distance_km', 'fee']

# 존재하는 컬럼들만 안전하게 드롭
drop_cols = [col for col in leakage_cols if col in merged.columns]
X = merged.drop(columns=drop_cols + ['is_overdue_100m', 'rental_id'])
y = merged['is_overdue_100m']

print(f"제거된 누수 컬럼: {drop_cols}")
print(f"남은 피처 컬럼 목록: {list(X.columns)}")

제거된 누수 컬럼: ['return_time', 'duration_min', 'distance_km', 'fee']
남은 피처 컬럼 목록: ['bike_id', 'user_id', 'rent_time', 'payment_method', 'station_id', 'bike_type', 'gear_count', 'manufacture_year', 'daily_fee', 'station_name', 'district', 'hour', 'weekday', 'dayofweek', 'join_date', 'membership_type', 'days_since_join']


## [STEP 4] 학습, 테스트 분할

### 요구사항

- **시점(`rent_time`) 기준**으로 분할하세요. 상위 80% 날짜를 기준선으로 삼습니다.
- 범주형 변수(`payment_method`, `membership_type` 등)를 인코딩하세요.
- **테스트 세트의 열 구성을 학습 세트에 맞추세요.**

⚠️

**랜덤 분할을 쓰면 안 됩니다.**

시계열 데이터에서 랜덤 분할을 쓰면, 미래의 패턴으로 과거를 예측하는 오류를 범하게 됩니다.


In [ ]:
# 1. rent_time 기준 정렬
merged = merged.sort_values('rent_time').reset_index(drop=True)

# 2. 상위 80% 시점을 기준(Split Point)으로 잡고 분할
split_idx = int(len(merged) * 0.8)

train_df = merged.iloc[:split_idx].copy()
test_df = merged.iloc[split_idx:].copy()

print(f"Train 셋 크기: {len(train_df):,}건 ({train_df['rent_time'].min()} ~ {train_df['rent_time'].max()})")
print(f"Test 셋 크기: {len(test_df):,}건 ({test_df['rent_time'].min()} ~ {test_df['rent_time'].max()})")

# 3. 누수 컬럼 및 타겟/식별자 정의 후 X, y 분리
leakage_cols = ['return_time', 'duration_min', 'distance_km', 'fee', 'is_overdue_100m', 'rental_id', 'rent_time']
features = [col for col in merged.columns if col not in leakage_cols]

X_train = train_df[features].copy()
y_train = train_df['is_overdue_100m'].copy()

X_test = test_df[features].copy()
y_test = test_df['is_overdue_100m'].copy()

# 4. 범주형 변수 인코딩 (One-Hot Encoding 적용)
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# 5. 테스트 세트의 열 구성을 학습 세트에 맞추기 (누락된 컬럼은 0으로 채우기)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(f"\n최종 Train 피처 수: {X_train_encoded.shape[1]}")
print(f"최종 Test 피처 수:  {X_test_encoded.shape[1]}")

Train 셋 크기: 10,569건 (2022-01-01 07:26:00 ~ 2022-10-20 16:02:00)
Test 셋 크기: 2,643건 (2022-10-20 16:18:00 ~ 2022-12-31 22:52:00)

최종 Train 피처 수: 2100
최종 Test 피처 수:  2100


## [STEP 5] 누수의 위력 확인

STEP 3에서 찾은 컬럼 중 하나(예: `fee` 또는 `duration_min`)를 **일부러 넣어서** 학습해 보세요.

### 요구사항

`RandomForestClassifier` 로 두 번 학습하고 비교하세요.

1. 누수 컬럼을 모두 **제거한** 피처
2. 누수 컬럼을 **포함한** 피처

정확도, 재현율, F1, AUC 를 나란히 놓고 비교합니다. 누수 컬럼이 포함되었을 때 모델 성능이 비정상적으로 완벽해지는지 확인하세요.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

# 1. 누수 컬럼을 포함하되, 숫자형/범주형 데이터만 선택하고 날짜/문자열 등 불필요한 타입 원천 차단
exclude_cols = ['is_overdue_100m', 'rental_id', 'rent_time', 'join_date', 'return_time']
leakage_included_cols = [col for col in merged.columns if col not in exclude_cols]

X_train_leak = train_df[leakage_included_cols].copy()
X_test_leak = test_df[leakage_included_cols].copy()

# 범주형 변수 인코딩 및 정렬
X_train_leak = pd.get_dummies(X_train_leak, drop_first=True)
X_test_leak = pd.get_dummies(X_test_leak, drop_first=True)
X_train_leak, X_test_leak = X_train_leak.align(X_test_leak, join='left', axis=1, fill_value=0)

# 혹시 남아있을지 모를 비수치형 컬럼 강제 변환 또는 제거
X_train_leak = X_train_leak.select_dtypes(include=['number', 'bool'])
X_test_leak = X_test_leak.select_dtypes(include=['number', 'bool'])

# 2. 모델 학습 (누수 포함)
rf_leak = RandomForestClassifier(random_state=42)
rf_leak.fit(X_train_leak, y_train)
y_pred_leak = rf_leak.predict(X_test_leak)
y_proba_leak = rf_leak.predict_proba(X_test_leak)[:, 1]

# 3. 모델 학습 (누수 제외 - 이전 단계의 X_train_encoded, X_test_encoded 사용)
X_train_clean = X_train_encoded.select_dtypes(include=['number', 'bool'])
X_test_clean = X_test_encoded.select_dtypes(include=['number', 'bool'])

rf_clean = RandomForestClassifier(random_state=42)
rf_clean.fit(X_train_clean, y_train)
y_pred_clean = rf_clean.predict(X_test_clean)
y_proba_clean = rf_clean.predict_proba(X_test_clean)[:, 1]

# 4. 성능 비교 지표 출력
results = {
    "Metrics": ["Accuracy", "Recall", "F1-Score", "ROC-AUC"],
    "Clean (누수 제거)": [
        accuracy_score(y_test, y_pred_clean),
        recall_score(y_test, y_pred_clean),
        f1_score(y_test, y_pred_clean),
        roc_auc_score(y_test, y_proba_clean)
    ],
    "Leakage (누수 포함)": [
        accuracy_score(y_test, y_pred_leak),
        recall_score(y_test, y_pred_leak),
        f1_score(y_test, y_pred_leak),
        roc_auc_score(y_test, y_proba_leak)
    ]
}

result_df = pd.DataFrame(results)
print(result_df.to_string(index=False))

DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int32DType'>, <class 'numpy.dtypes.Int32DType'>, <class 'numpy.dtypes.Int32DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.BoolDType'>, <class 'numpy.dtypes.Boo